# Dia 2 — Memória Persistente + Múltiplas Tools + Middleware

No Dia 1 construímos um agente que funcionava — mas ele era **sem memória**: cada conversa começava do zero, sem saber nada do que havia sido dito antes.

Hoje resolvemos isso e adicionamos mais duas capacidades:

1. **Memória persistente** com `SqliteSaver` — o agente lembra conversas anteriores entre sessões, mesmo que o ambiente reinicie
2. **Mais tools de e-mail** — além de enviar, o agente agora consegue ler, buscar e listar e-mails recebidos e enviados
3. **Middleware HITL** — o agente pausa antes de executar tools sensíveis e aguarda aprovação humana

> 💡 **Conceito central do dia:** *`thread_id`* — um identificador de sessão. Cada usuário tem seu próprio histórico isolado no mesmo banco de dados. Mudar o `thread_id` é como trocar de pessoa.

Este notebook é **autossuficiente** — você não precisa do notebook do Dia 1 aberto.

| Parte | Tema |
|---|---|
| Setup | Dependências, credenciais, LLM e e-mail |
| Memória | `SqliteSaver` + `invocar_com_memoria()` |
| Tools | 5 tools de e-mail — definição e teste direto |
| Agente | Um único agente com todas as tools |
| Sem memória | `agente_sem_checkpointer` — contraste direto com o agente com memória |
| Middleware | `HumanInTheLoopMiddleware` — aprovar, rejeitar ou editar antes de executar |

---

## Setup

In [ ]:
!pip install -q langchain-anthropic langchain langgraph langgraph-checkpoint-sqlite aiosqlite

Preencha com os dados fornecidos pelo professor. `PROXY_URL` + `ALUNO_TOKEN` para o proxy LLM; `EMAIL_URL` + `EMAIL_TOKEN` + `EMAIL_PASSWORD` para o servidor de e-mail.

In [3]:
# Preencha com os valores que o professor forneceu
PROXY_URL      = "https://interview-server-mocado.b60gda.easypanel.host/"
ALUNO_TOKEN    = "xpto_aluno-01"   # token do proxy LLM

EMAIL_URL      = "https://interview-email-server.b60gda.easypanel.host/"
EMAIL_TOKEN    = "aluno-01"
EMAIL_PASSWORD = "1234"

print("Credenciais carregadas.")

Credenciais carregadas.


Verificamos se o proxy LLM está respondendo antes de continuar.

In [4]:
import requests

r = requests.get(f"{PROXY_URL}/health")
print(r.status_code, r.json())

200 {'status': 'ok', 'anthropic_key_configured': True, 'alunos_registrados': 10}


Configuramos o LLM com `max_tokens=512` — mais espaço que no Dia 1 para acomodar respostas com listas de e-mails.

In [5]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=ALUNO_TOKEN,
    base_url=PROXY_URL,
    max_tokens=512,
)

resposta = llm.invoke([HumanMessage(content="Responda apenas: conexão ok!")])
print(resposta.content)

conexão ok!


Fazemos login no servidor de e-mail para obter o token de autenticação que todas as tools usarão.

In [6]:
r = requests.post(f"{EMAIL_URL}/auth/login", json={
    "token": EMAIL_TOKEN,
    "password": EMAIL_PASSWORD,
})
assert r.status_code == 200, f"Login falhou: {r.json()}"
_headers = {"Authorization": f"Bearer {EMAIL_TOKEN}"}
print("Login OK →", r.json())

Login OK → {'token': 'aluno-01', 'name': 'Aluno 1', 'email': 'aluno01@curso.ia'}


Verificamos a caixa de entrada diretamente via API — sem agente — para confirmar que o login funcionou e ver o estado inicial dos e-mails.

In [24]:
# Verificar caixa de entrada diretamente (sem agente)
r = requests.get(f"{EMAIL_URL}/emails/inbox", headers=_headers)
inbox = r.json()
print(f"{inbox['count']} mensagem(ns) para {inbox['email']}")
for msg in inbox["messages"]:
    print(f"  [{msg['id']}] De: {msg['from']} | {msg['subject']}")

4 mensagem(ns) para aluno01@curso.ia
  [26a8891f-b176-4d10-8954-6dc2f3f8e1ca] De: nao_responda@curso.ia | [PRODUÇÃO] Meta atingida — Linha 3
  [0a431337-38df-42c5-a589-556e8d7327e4] De: nao_responda@curso.ia | [FALHA] Sistema de Pagamentos
  [9fe1c2d6-11bb-4f88-a7b3-8fb7a7dbca38] De: nao_responda@curso.ia | [FALHA] Sistema de pagamentos
  [aba78f56-f017-4278-a854-c4d4a513e738] De: nao_responda@curso.ia | [PRODUÇÃO] Meta atingida — Linha 3


---
## Memória Persistente — SqliteSaver

Configuramos a memória **antes** de criar qualquer agente. Assim todos os agentes nascem com ela.

> 💡 `SqliteSaver` grava o histórico em um arquivo `.db`. No Google Colab, salvamos no **Google Drive** — a memória sobrevive mesmo que o ambiente reinicie.

O `thread_id` é o identificador da sessão — funciona como um "nome de usuário" para o checkpointer:

- **Mesmo `thread_id`** → o agente retoma a conversa de onde parou
- **`thread_id` diferente** → nova conversa, histórico zerado
- Na prática: use o token ou e-mail do usuário como `thread_id` para separar sessões por pessoa

In [10]:
import os
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/curso_agentes", exist_ok=True)
    DB_PATH = "/content/drive/MyDrive/curso_agentes/memoria.db"
    print(f"Google Drive montado. Banco: {DB_PATH}")
except ImportError:
    DB_PATH = "memoria.db"
    print(f"Executando localmente. Banco: {DB_PATH}")

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(conn)

def invocar_com_memoria(prompt: str, thread_id: str = None) -> str:
    """Invoca o agente com memória persistente.

    thread_id identifica a sessão — por padrão usa o token do aluno.
    Mesma thread retoma de onde parou; thread diferente = histórico zerado.
    """
    tid = thread_id or EMAIL_TOKEN
    resultado = agente.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"configurable": {"thread_id": tid}, "recursion_limit": 10},
    )
    return resultado["messages"][-1].content

print("checkpointer e invocar_com_memoria() prontos.")

Executando localmente. Banco: memoria.db
checkpointer e invocar_com_memoria() prontos.


---
## Tools de e-mail

Cinco tools que o agente poderá usar. A **docstring** de cada uma é o que o modelo lê para decidir quando e como usá-la.

Testamos cada tool **diretamente** (sem agente) para confirmar que funciona antes de conectar.

In [11]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from typing import Optional

@tool
def send_email(to: str, subject: str, body: str, cc: Optional[str] = None) -> str:
    """Envia um e-mail pelo servidor do curso.

    Use esta tool quando o usuário pedir para enviar ou responder um e-mail.

    Args:
        to: E-mail do destinatário (ex: aluno02@curso.ia)
        subject: Assunto do e-mail
        body: Corpo completo do e-mail
        cc: E-mails em cópia, separados por vírgula (opcional)
    """
    payload = {"to": to, "subject": subject, "body": body}
    if cc:
        payload["cc"] = cc
    r = requests.post(f"{EMAIL_URL}/emails/send", headers=_headers, json=payload)
    if r.status_code in (200, 201):
        return f"E-mail enviado com sucesso para '{to}'."
    return f"Erro ao enviar ({r.status_code}): {r.json()}"

print("Tool send_email definida.")

Tool send_email definida.


`get_inbox` não tem argumentos — retorna toda a caixa de entrada. O agente decide quando chamá-la com base na docstring.

In [12]:
@tool
def get_inbox() -> str:
    """Lista os e-mails na caixa de entrada do usuário.

    Retorna id, data, remetente e assunto de cada mensagem.
    Use esta tool quando o usuário quiser saber o que tem na caixa de entrada.
    """
    r = requests.get(f"{EMAIL_URL}/emails/inbox", headers=_headers)
    inbox = r.json()
    if inbox["count"] == 0:
        return "A caixa de entrada está vazia."
    linhas = [f"Total: {inbox['count']} mensagem(ns)\n"]
    for msg in inbox["messages"]:
        linhas.append(
            f"ID: {msg['id']} | Data: {msg['timestamp']} | "
            f"De: {msg['from']} | Assunto: {msg['subject']}"
        )
    return "\n".join(linhas)

print(get_inbox.invoke({}))

Total: 4 mensagem(ns)

ID: 26a8891f-b176-4d10-8954-6dc2f3f8e1ca | Data: 2026-04-23T15:15:08.580370+00:00 | De: nao_responda@curso.ia | Assunto: [PRODUÇÃO] Meta atingida — Linha 3
ID: 0a431337-38df-42c5-a589-556e8d7327e4 | Data: 2026-04-23T15:08:30.530734+00:00 | De: nao_responda@curso.ia | Assunto: [FALHA] Sistema de Pagamentos
ID: 9fe1c2d6-11bb-4f88-a7b3-8fb7a7dbca38 | Data: 2026-04-23T15:07:53.364564+00:00 | De: nao_responda@curso.ia | Assunto: [FALHA] Sistema de pagamentos
ID: aba78f56-f017-4278-a854-c4d4a513e738 | Data: 2026-04-23T15:00:08.440844+00:00 | De: nao_responda@curso.ia | Assunto: [PRODUÇÃO] Meta atingida — Linha 3


`get_sent` lista os e-mails já enviados pelo usuário — útil para confirmar envios ou evitar duplicatas.

In [13]:
@tool
def get_sent() -> str:
    """Lista os e-mails enviados pelo usuário.

    Retorna id, data, destinatário e assunto de cada mensagem enviada.
    Use esta tool quando o usuário quiser ver os e-mails que já enviou.
    """
    r = requests.get(f"{EMAIL_URL}/emails/sent", headers=_headers)
    sent = r.json()
    if sent["count"] == 0:
        return "Nenhum e-mail enviado ainda."
    linhas = [f"Total: {sent['count']} mensagem(ns) enviada(s)\n"]
    for msg in sent["messages"]:
        destinatarios = ", ".join(msg["to"]) if isinstance(msg["to"], list) else msg["to"]
        linhas.append(
            f"ID: {msg['id']} | Data: {msg['timestamp']} | "
            f"Para: {destinatarios} | Assunto: {msg['subject']}"
        )
    return "\n".join(linhas)

print(get_sent.invoke({}))

Total: 17 mensagem(ns) enviada(s)

ID: d528d541-2737-4af8-abbe-b2e2e8e549cc | Data: 2026-04-24T01:39:12.157949+00:00 | Para: aluno03@curso.ia | Assunto: Reunião
ID: dbc965d1-a2ee-4f87-81df-d38df91403f5 | Data: 2026-04-24T01:33:24.105047+00:00 | Para: aluno03@curso.ia | Assunto: Reunião
ID: c55670d5-ba15-4b06-9657-13b6aedd6663 | Data: 2026-04-24T01:27:53.561740+00:00 | Para: aluno03@curso.ia | Assunto: Middleware
ID: f35dfb2f-55dc-478d-b90b-e0ef8637a4f7 | Data: 2026-04-24T01:13:47.565182+00:00 | Para: aluno05@curso.ia | Assunto: Confirmação de Reunião
ID: 99e3ab27-0217-4fb0-8ec8-47b2006b64b6 | Data: 2026-04-24T01:10:04.698011+00:00 | Para: aluno05@curso.ia | Assunto: Desmarcação de Reunião
ID: 46f0c307-a9bc-434d-bbeb-7d5d9188abe9 | Data: 2026-04-24T01:09:59.986709+00:00 | Para: aluno05@curso.ia | Assunto: Confirmação de Reunião
ID: de0239ea-17ca-4ea9-b241-0c7e793914b9 | Data: 2026-04-24T00:00:29.314494+00:00 | Para: aluno05@curso.ia | Assunto: Confirmação de Reunião
ID: 9a18a19f-3c64-43

`get_email` lê o conteúdo completo de uma mensagem pelo `id`. A docstring instrui o agente a obter o ID com `get_inbox` primeiro, quando necessário.

In [14]:
@tool
def get_email(email_id: str) -> str:
    """Lê o conteúdo completo de um e-mail específico.

    Use esta tool quando o usuário quiser ler ou responder um e-mail.
    Você precisa do ID — obtenha-o com get_inbox primeiro.

    Args:
        email_id: ID do e-mail a ser lido.
    """
    r = requests.get(f"{EMAIL_URL}/emails/{email_id}", headers=_headers)
    if r.status_code == 404:
        return f"E-mail com ID '{email_id}' não encontrado."
    msg = r.json()
    return (
        f"De:      {msg['from']}\n"
        f"Para:    {msg.get('to', '')}\n"
        f"CC:      {msg.get('cc', '')}\n"
        f"Assunto: {msg['subject']}\n"
        f"Data:    {msg['timestamp']}\n"
        f"\n{msg['body']}"
    )

# Teste: substitua pelo ID de um e-mail real
# print(get_email.invoke({"email_id": "COLE_UM_ID_AQUI"}))
print("Tool get_email definida.")

Tool get_email definida.


`search_emails` filtra e-mails por remetente ou palavras no assunto — sem precisar do ID. O agente a usa quando o usuário quer buscar por contexto.

In [15]:
@tool
def search_emails(query: str) -> str:
    """Busca e-mails na caixa de entrada por remetente ou palavras no assunto.

    Use esta tool quando o usuário perguntar sobre e-mails de uma pessoa
    específica ou com um assunto específico.

    Args:
        query: Termo de busca — nome, e-mail do remetente ou palavra no assunto.
    """
    r = requests.get(f"{EMAIL_URL}/emails/inbox", headers=_headers)
    inbox = r.json()
    q = query.lower()
    encontrados = [
        msg for msg in inbox["messages"]
        if q in msg["from"].lower() or q in msg["subject"].lower()
    ]
    if not encontrados:
        return f"Nenhum e-mail encontrado para a busca: '{query}'."
    linhas = [f"{len(encontrados)} e-mail(s) encontrado(s) para '{query}':\n"]
    for msg in encontrados:
        linhas.append(f"ID: {msg['id']} | De: {msg['from']} | Assunto: {msg['subject']}")
    return "\n".join(linhas)

print(search_emails.invoke({"query": "reunião"}))

Nenhum e-mail encontrado para a busca: 'reunião'.


---
## Agente

Um único agente com todas as 5 tools e memória persistente via `checkpointer`.

In [16]:
todas_as_tools = [get_inbox, get_sent, get_email, search_emails, send_email]

agente = create_agent(
    model=llm,
    tools=todas_as_tools,
    system_prompt=(
        "Você é um assistente de e-mail. "
        "Use as ferramentas disponíveis para ler, buscar, listar enviados e enviar e-mails. "
        "Os endereços seguem o padrão alunoXX@curso.ia (ex: aluno02@curso.ia). "
        "Seja objetivo nas respostas."
    ),
    # OBSERVE A MEMORIA AQUI VIA CHECKPOINTER
    checkpointer=checkpointer,
)

print(f"Agente criado com {len(todas_as_tools)} tools:", [t.name for t in todas_as_tools])

Agente criado com 5 tools: ['get_inbox', 'get_sent', 'get_email', 'search_emails', 'send_email']


## Testes com o Agente com memoria  -  `invocar_com_memoria()`

In [32]:
# Teste 1 — listar inbox
print(invocar_com_memoria("O que tem na minha caixa de entrada?"))

Sua caixa de entrada tem **4 e-mails**:

1. **[PRODUÇÃO] Meta atingida — Linha 3** - De: nao_responda@curso.ia (23/04/2026 às 15:15:08)
2. **[FALHA] Sistema de Pagamentos** - De: nao_responda@curso.ia (23/04/2026 às 15:08:30)
3. **[FALHA] Sistema de pagamentos** - De: nao_responda@curso.ia (23/04/2026 às 15:07:53)
4. **[PRODUÇÃO] Meta atingida — Linha 3** - De: nao_responda@curso.ia (23/04/2026 às 15:00:08)

Todos são notificações do sistema. 📧


Buscamos e-mails por assunto — o agente decide usar `search_emails` ao detectar que o usuário quer filtrar por conteúdo, não por identificador.

In [33]:
# Teste 2 — busca por assunto
print(invocar_com_memoria("Tem algum e-mail sobre reunião?"))

Não encontrei nenhum e-mail sobre "reunião" na sua caixa de entrada. 📧


Encadeamento de duas tools: o agente precisa primeiro buscar o e-mail e depois ler o conteúdo completo — observe a sequência de tool calls na resposta.

In [34]:
# Teste 3 — sequência de tools (search → get_email)
print(invocar_com_memoria("Encontre e-mails do aluno05 e me diga o que o último diz."))

Não encontrei nenhum e-mail do aluno05 na sua caixa de entrada. 📧

Nota: Os e-mails que você enviou para aluno05@curso.ia estão na sua caixa de saída, mas ele ainda não respondeu.


Testamos o envio com memória ativa. O histórico da thread acumula — os próximos testes herdam o contexto desta conversa.

In [35]:
# Teste 1 — listar inbox
print(invocar_com_memoria("Envie um email par ao aluno05 confirmando nossa reunião para aamnhã as 10:00."))

Pronto! ✅ E-mail enviado com sucesso para aluno05@curso.ia confirmando a reunião para amanhã às 10:00.


Na mesma thread, pedimos ao agente para desfazer a ação anterior. Com memória, ele sabe o destinatário e o contexto sem precisarmos repetir.

In [ ]:
# Teste 2 — busca por assunto
print(invocar_com_memoria("Acabei de lembrar que tenho compromisso, vamos desmarcar a reunião."))

Sua última pergunta foi: "Acabei de lembrar que tenho compromisso, vamos desmarcar a reunião."

E eu enviei um e-mail para aluno05@curso.ia desmarcando a reunião! 😊


O modo verbose imprime todas as mensagens internas: `HumanMessage`, `AIMessage` (com tool calls) e `ToolMessage` (resposta da tool). É o ciclo **Raciocínio → Ação → Observação** do agente.

In [37]:
# Verbose — ver o chain-of-thought interno
# Mostra TODAS as mensagens trocadas, incluindo chamadas de tool e respostas.

def invocar_verbose(prompt, ag=agente):
    resultado = ag.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 10},
    )
    for msg in resultado["messages"]:
        tipo = msg.__class__.__name__
        print(f"\n{'='*40}")
        print(f"  {tipo}")
        print(f"{'='*40}")
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  → tool: {tc['name']}")
                print(f"     args: {tc['args']}")
        elif hasattr(msg, "content"):
            conteudo = str(msg.content)
            print(conteudo[:400] + ("..." if len(conteudo) > 400 else ""))

invocar_verbose("Tem algum e-mail sobre reunião? Se sim, me diga o remetente.")


  HumanMessage
Tem algum e-mail sobre reunião? Se sim, me diga o remetente.

  AIMessage
  → tool: search_emails
     args: {'query': 'reunião'}

  ToolMessage
Nenhum e-mail encontrado para a busca: 'reunião'.

  AIMessage
Não encontrei nenhum e-mail sobre reunião na sua caixa de entrada.


---
## Sem memória — `agente_sem_checkpointer`

Para demonstrar o papel do checkpointer de forma direta, criamos um **segundo agente idêntico — mas sem `checkpointer`**.

Sem ele, o agente não tem onde guardar o histórico. Cada chamada começa com o State vazio, independente do que aconteceu antes.

| | `agente` | `agente_sem_checkpointer` |
|---|---|---|
| Tools | as mesmas 5 | as mesmas 5 |
| System prompt | idêntico | idêntico |
| Checkpointer | `SqliteSaver` | nenhum |
| Lembra entre chamadas? | ✅ sim | ❌ não |

In [17]:
import uuid

agente_sem_checkpointer = create_agent(
    model=llm,
    tools=todas_as_tools,
    system_prompt=(
        "Você é um assistente de e-mail. "
        "Use as ferramentas disponíveis para ler, buscar, listar enviados e enviar e-mails. "
        "Os endereços seguem o padrão alunoXX@curso.ia (ex: aluno02@curso.ia). "
        "Seja objetivo nas respostas."
    ),
    # sem checkpointer — o agente não tem onde salvar o histórico
)

def invocar_sem_memoria(prompt: str) -> str:
    """Invoca o agente sem checkpointer — cada chamada começa com o State vazio."""
    resultado = agente_sem_checkpointer.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config={"recursion_limit": 10},
    )
    return resultado["messages"][-1].content

print("agente_sem_checkpointer e invocar_sem_memoria() prontos.")

agente_sem_checkpointer e invocar_sem_memoria() prontos.


Enviamos o mesmo e-mail de confirmação usando o agente sem checkpointer — funciona normalmente para ações isoladas.

In [40]:
# Teste 1 — listar inbox
print(invocar_sem_memoria("Envie um email par ao aluno05 confirmando nossa reunião para aamnhã as 10:00."))

Feito! E-mail enviado para aluno05@curso.ia confirmando a reunião de amanhã às 10:00.


Pedimos algo que dependeria de contexto anterior. Sem memória, o agente não consegue inferir o destinatário e pede informações adicionais.

In [41]:
# Teste 2 — busca por assunto
print(invocar_sem_memoria("Acabei de lembrar que tenho compromisso, vamos desmarcar a reunião."))

Entendi que você quer desmarcar uma reunião. Para ajudar você adequadamente, preciso:

1. **Localizar o e-mail sobre a reunião** - Preciso verificar sua caixa de entrada ou enviados
2. **Identificar o destinatário** - Quem você precisa informar sobre o desmarcamento

Você pode me informar:
- **Quem é a pessoa** (ou pessoas) envolvidas na reunião?
- **Qual é o assunto/tema** da reunião que precisa desmarcar?

Assim poderei encontrar o e-mail relevante e ajudar você a enviar uma mensagem de desmarcamento.


---
## Middleware — Aprovação antes de executar uma tool

`create_agent` aceita um parâmetro `middleware` — uma lista de interceptadores que atuam em momentos específicos da execução do agente.

### Tipos de middleware

| Tipo | O que faz | Pronto para usar? |
|---|---|---|
| `HumanInTheLoopMiddleware` | Pausa antes de executar uma tool e aguarda decisão humana | ✅ sim |
| Middleware customizado | Logging, retry, rate limiting, transformação de args — você implementa | 🔧 manual |





> **Diferença em relação ao LangGraph `interrupt()`:**  
> O middleware é **declarativo** — você lista quais tools precisam de aprovação, sem construir um grafo.  
> O `interrupt()` do LangGraph é **imperativo** — você decide exatamente onde pausar no fluxo.  
> Ambos resolvem o mesmo problema; middleware é mais simples quando você usa `create_agent`.

### Exemplo — middleware customizado com logging de horário

Antes de ver o HITL, aqui está um middleware customizado simples: ele loga o horário exato antes e depois de cada chamada ao LLM, usando os hooks `before_model` e `after_model`.

O middleware customizado usa **hooks** que o LangChain chama em pontos específicos:

| Hook | Quando é chamado |
|---|---|
| `before_agent` | Antes do agente processar a mensagem |
| `before_model` | Antes de chamar o LLM |
| `after_model` | Após a resposta do LLM |
| `wrap_tool_call` | Envolve a execução de cada tool (antes + depois) |
| `after_agent` | Após o agente finalizar |

In [18]:
from datetime import datetime
from langchain.agents.middleware import AgentMiddleware

class LogMiddleware(AgentMiddleware):
    """Middleware customizado que loga o horário de cada chamada ao LLM."""

    def before_model(self, state, config):
        print(f"[{datetime.now().strftime('%H:%M:%S.%f')[:-3]}] → LLM chamado")
        return {}

    def after_model(self, state, config):
        print(f"[{datetime.now().strftime('%H:%M:%S.%f')[:-3]}] ← LLM respondeu")
        return {}

agente_com_log = create_agent(
    model=llm,
    tools=todas_as_tools,
    system_prompt=(
        "Você é um assistente de e-mail. "
        "Os endereços seguem o padrão alunoXX@curso.ia. "
        "Seja objetivo."
    ),
    middleware=[LogMiddleware()],
)



### Invoke que chama tool (imprime midleware duas vezes)

In [49]:
resultado = agente_com_log.invoke(
    {"messages": [{"role": "user", "content": "O que tem na minha caixa de entrada?"}]},
    config={"recursion_limit": 10},
)
print(resultado["messages"][-1].content)

[21:20:59.681] → LLM chamado
[21:21:00.767] ← LLM respondeu
[21:21:01.179] → LLM chamado
[21:21:02.737] ← LLM respondeu
Você tem **4 mensagens** na caixa de entrada:

1. **[PRODUÇÃO] Meta atingida — Linha 3** | 23/04/2026 às 15:15 | De: nao_responda@curso.ia
2. **[FALHA] Sistema de Pagamentos** | 23/04/2026 às 15:08 | De: nao_responda@curso.ia
3. **[FALHA] Sistema de pagamentos** | 23/04/2026 às 15:07 | De: nao_responda@curso.ia
4. **[PRODUÇÃO] Meta atingida — Linha 3** | 23/04/2026 às 15:00 | De: nao_responda@curso.ia

Quer ler alguma delas?


### Invoke que não chama tool (imprime midleware uma vez)

In [50]:
resultado = agente_com_log.invoke(
    {"messages": [{"role": "user", "content": "Ola, tudo bem?"}]},
    config={"recursion_limit": 10},
)
print(resultado["messages"][-1].content)

[21:21:29.063] → LLM chamado
[21:21:30.799] ← LLM respondeu
Olá! Tudo bem, sim! 😊

Sou seu assistente de e-mail. Posso ajudá-lo com:

- 📥 **Verificar sua caixa de entrada**
- 📤 **Ver e-mails que você enviou**
- 🔍 **Buscar e-mails específicos**
- 📧 **Ler e-mails completos**
- ✍️ **Enviar ou responder e-mails**

O que você gostaria de fazer?



### Human-in-the-Loop — aprovação antes de executar a tool

Agora o caso mais comum de middleware em produção: pausar o agente **antes** de executar uma tool sensível e aguardar uma decisão humana.

Criamos um `agente_hitl` com `HumanInTheLoopMiddleware` configurado para interceptar `send_email`. A cada tentativa de envio, o agente pausa e devolve um `__interrupt__` com os argumentos que usaria. A execução só continua quando chamamos `agent.invoke(Command(resume=...))` com uma das três decisões:

| Decisão | Efeito |
|---|---|
| `approve` | Tool executa com os argumentos originais |
| `reject` | Tool é cancelada — agente recebe o motivo e informa o usuário |
| `edit` | Argumentos são substituídos antes de executar |





*O `checkpointer` é **obrigatório** para HITL — sem ele o agente não consegue retomar de onde parou.

In [20]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

agente_hitl = create_agent(
    model=llm,
    tools=todas_as_tools,
    system_prompt=(
        "Você é um assistente de e-mail. "
        "Use as ferramentas disponíveis para ler, buscar e enviar e-mails. "
        "Os endereços seguem o padrão alunoXX@curso.ia. "
        "Seja objetivo nas respostas. "
        "Se uma ação for recusada por um humano (mensagem contendo 'rejected' ou 'reject'), "
        "informe o usuário de forma clara que o envio foi bloqueado e apresente o motivo recebido."
    ),
    checkpointer=checkpointer,  # obrigatório para HITL
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": {"allowed_decisions": ["approve", "edit", "reject"]},
            }
        )
    ],
)

print("Agente com middleware HITL criado.")

Agente com middleware HITL criado.


Execute a célula abaixo. O agente vai pausar e aparecer um campo de input — escolha:

- **`a`** — aprovar (e-mail é enviado normalmente)
- **`r`** — rejeitar (você digita o motivo, agente informa o usuário)
- **`e`** — editar (você corrige os campos antes de enviar)

Execute mais de uma vez, tentando as três opções.

In [ ]:
from langgraph.types import Command

config_hitl = {"configurable": {"thread_id": f"hitl-{uuid.uuid4()}"}, "recursion_limit": 10}

resultado = agente_hitl.invoke(
    {"messages": [{"role": "user", "content":
        "Manda um e-mail para aluno03@curso.ia com assunto 'Reunião' e corpo 'Podemos marcar para sexta às 14h?'"}]},
    config=config_hitl,
)

interrupt_info = resultado.get("__interrupt__")

if not interrupt_info:
    # Nenhuma tool sensível envolvida — resposta direta do agente
    print(resultado["messages"][-1].content)

else:
    req = interrupt_info[0].value["action_requests"][0]
    tool_name = req["name"]
    tool_args = req["args"]

    print(f"\n⏸️  O agente quer executar: {tool_name}")
    print("   Argumentos:")
    for k, v in tool_args.items():
        print(f"     {k}: {v}")

    decisao = input("\nDecisão — [a]provar / [r]ejeitar / [e]ditar: ").strip().lower()

    if decisao == "r":
        motivo = input("Motivo da rejeição: ")
        agente_hitl.invoke(
            Command(resume={"decisions": [{"type": "reject", "feedback": motivo}]}),
            config=config_hitl,
        )
        print(f"\n❌ Envio bloqueado.")
        print(f"   Motivo: {motivo}")

    elif decisao == "e":
        print("Edite os argumentos (Enter para manter o valor atual):")
        novos_args = {}
        for k, v in tool_args.items():
            novo = input(f"  {k} [{v}]: ").strip()
            novos_args[k] = novo if novo else v
        agente_hitl.invoke(
            Command(resume={"decisions": [{
                "type": "edit",
                "edited_action": {"name": tool_name, "args": novos_args},
            }]}),
            config=config_hitl,
        )
        print(f"\n✏️  E-mail editado e enviado.")
        print(f"   Para:    {novos_args.get('to')}")
        print(f"   Assunto: {novos_args.get('subject')}")
        print(f"   Corpo:   {novos_args.get('body')}")

    else:
        resultado = agente_hitl.invoke(
            Command(resume={"decisions": [{"type": "approve"}]}),
            config=config_hitl,
        )
        print("\n✅ Aprovado. Resposta final:")
        print(resultado["messages"][-1].content)

Olá! Tudo bem sim! 👋

Sou seu assistente de e-mail. Posso ajudá-lo com:

- **Ler e-mails** da sua caixa de entrada
- **Buscar e-mails** específicos por remetente ou assunto
- **Enviar ou responder** e-mails
- **Ver e-mails enviados**

O que você gostaria de fazer?


---
## Resumo do Dia 2

| Conceito | O que aprendemos |
|---|---|
| `SqliteSaver` + Google Drive | Memória que persiste entre sessões |
| `thread_id` | Identifica a sessão — mesmo token, mesmo histórico; token diferente, histórico zerado |
| `invocar_com_memoria()` | Invoca com histórico persistente por `thread_id` |
| `agente_sem_checkpointer` | Agente idêntico mas sem checkpointer — State vazio a cada chamada |
| `invocar_sem_memoria()` | Demonstra ausência de memória de forma explícita |
| `@tool` + docstring | A docstring é o que o modelo lê para decidir quando usar a tool |
| Agente multi-tool | O modelo escolhe a tool certa dado o contexto do pedido |
| `HumanInTheLoopMiddleware` | Intercepta tool calls antes de executar — sem construir um grafo |
| `approve` / `reject` / `edit` | Três decisões possíveis ao receber um interrupt de middleware |
| `Command(resume=...)` | Retoma o agente pausado com a decisão humana |

### Middleware vs LangGraph `interrupt()`

| | Middleware (`create_agent`) | `interrupt()` (LangGraph) |
|---|---|---|
| Como configurar | Declarativo — lista as tools | Imperativo — coloca no nó certo |
| Quando usar | Aprovação simples por tool | Fluxos complexos com bifurcações |
| Precisa de grafo? | Não | Sim |

### O que vem no Dia 3

- Introdução ao RAG com FAISS
- O agente consulta uma base de conhecimento externa antes de responder